# 01 - Filtrar Chihuahua desde el dataset nacional consolidado
Insumo: data/processed/suicidio_2019_2024_consolidado.csv del repo
mexico-suicide-data-curation (49,918 registros, ya validado).

DECISION METODOLOGICA VALIDADA: se filtra por Ent_resid (residencia habitual),
NO por Ent_ocurr (lugar de ocurrencia). Confirmado contra cifra de prensa
2023 (561 casos): Ent_resid=='08' da 554 (1.25% diff), Ent_ocurr=='08' da 512
(8.7% diff). Ademas, los denominadores poblacionales de CONAPO son por
residencia, asi que numerador y denominador deben usar el mismo criterio
(ver docs/methodology.md).

Este notebook asume que copiaste ese archivo a data/raw/ de este repo,
o ajustas la ruta abajo para apuntar al repo original en tu maquina.

In [ ]:
import pandas as pd

# Ajustar ruta segun donde tengas el consolidado nacional
RUTA_CONSOLIDADO_NACIONAL = '../data/raw/suicidio_2019_2024_consolidado.csv'

df_nacional = pd.read_csv(RUTA_CONSOLIDADO_NACIONAL, encoding='utf-8', low_memory=False, dtype=str)
print(f'Registros nacionales: {len(df_nacional):,}')


## 1. Filtrar a Chihuahua por RESIDENCIA HABITUAL (Ent_resid == '08')
No usar Ent_ocurr aqui (ver nota metodologica arriba).

In [ ]:
df_chih = df_nacional[df_nacional['Ent_resid'] == '08'].copy()
print(f'Registros de Chihuahua (por residencia habitual): {len(df_chih):,}')
df_chih['anio_dataset'].value_counts().sort_index()


## 2. Verificacion contra cifras publicadas
Cifras de referencia encontradas en prensa/INEGI (por residencia habitual;
verificar y documentar fuente exacta en docs/methodology.md antes de usar
en el articulo):
- 2022: tasa 11.2 (1er lugar nacional)
- 2023: 554 casos propios vs. 561 de prensa (1.25% diff) -- VALIDADO
- 2024: tasa 16.4 (1er lugar nacional)

In [ ]:
casos_chih_por_anio = df_chih['anio_dataset'].value_counts().sort_index()
print('Casos Chihuahua por anio (residencia habitual, pipeline propio):')
print(casos_chih_por_anio)
print()
diferencia_2023 = abs(casos_chih_por_anio.get('2023', casos_chih_por_anio.get(2023)) - 561)
print(f'2023: diferencia vs. cifra de prensa (561) = {diferencia_2023} casos '
      f'({round(diferencia_2023/561*100, 2)}%)')


## 3. Casos por municipio de RESIDENCIA (conteo crudo, SIN tasa todavia)
OJO: esta tabla por si sola es enganosa (favorece a municipios grandes).
Sirve solo para inspeccion inicial, no para el articulo final. Se calculan
tasas en el siguiente notebook, una vez unida la poblacion de CONAPO.

In [ ]:
casos_por_municipio = df_chih.groupby(['anio_dataset', 'Mun_resid']).size().reset_index(name='casos')
casos_por_municipio.sort_values(['anio_dataset', 'casos'], ascending=[True, False]).head(20)


## 4. Guardar subconjunto de Chihuahua
Handoff a 02_population_merge.ipynb: unir con poblacion municipal CONAPO
(por residencia) para calcular tasas.

In [ ]:
import os
os.makedirs('../data/processed', exist_ok=True)
df_chih.to_csv('../data/processed/suicidio_chihuahua_2019_2024.csv', index=False, encoding='utf-8')
print(f'Guardado: {len(df_chih):,} registros')


## 5. Hallazgos
- Filtro por Ent_resid confirmado como correcto (1.25% diff vs. prensa 2023),
  muy superior al filtro por Ent_ocurr (8.7% diff). Documentado en
  docs/methodology.md.
- _Documentar aqui cualquier patron inicial observado por municipio antes de
  calcular tasas._